# Pipeline

            ┌───────────────────────────────┐
            │        Input PDF Reports      │
            │   (IR Report / Governance)    │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 1. Page Topic Extraction      │
            │   (Ollama - Mini Model)       │
            │ - Read PDF page-by-page       │
            │ - Extract main topics         │
            │ - Build document page tree    │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 2. Indicator Page Isolation   │
            │   (Gemini - Large Model)      │
            │ - Match indicators to pages   │
            │ - Return relevant page lists  │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 3. Focused Page Expansion     │
            │   (±1 Page Window)            │
            │ - Expand candidate pages      │
            │ - Deduplicate page set        │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 4. Page Content Re-parsing    │
            │   (Docling → Markdown Export) │
            │ - Convert selected pages      │
            │ - Preserve table structure    │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 5. Indicator Evidence Filter  │
            │   (Gemini Decision per Page)  │
            │ - Extract only relevant text  │
            │ - Return null if unrelated    │
            └───────────────┬───────────────┘
                            │
                            ▼
            ┌───────────────────────────────┐
            │ 6. Final Answer Generation    │
            │   (Gemini Context QA)         │
            │ - Use extracted evidence only │
            │ - Output "NO INFORMATION"     │
            │   if missing                  │
            └───────────────────────────────┘


In [14]:
from ollama import chat
import json
import re
import fitz
from tqdm import tqdm
import os
import google.generativeai as genai
import time
from dotenv import load_dotenv
import warnings
import requests
from urllib.parse import urlparse
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

# Load environment variables FIRST
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError(
        "❌ GOOGLE_API_KEY not found in environment variables.\n"
        "   Please set it in .env file or export it:\n"
        "   export GOOGLE_API_KEY='your-api-key'"
    )

genai.configure(api_key=GOOGLE_API_KEY)
print(f"✅ Google API configured successfully")

warnings.filterwarnings("ignore")

class PDFProcessor:
    def __init__(self, model_name: str = "qwen2.5:1.5b"):
        self.model_name = model_name
        self.SYSTEM_PROMPT = """
        You read ONE page of a financial report.

        Task:
        List the main topics discussed on this page.

        Rules:
        - Topics must be short noun phrases.
        - Only include topics clearly mentioned.
        - No explanations.

        Output:
        Return ONLY a JSON array of strings.
        """

    def extract_features_clean(self, text: str) -> list[str]:
        return list(dict.fromkeys(s.strip() for s in re.findall(r'"([^"]+)"', text)))

    def extract_page_topics(
        self,
        page_content: str,
    ) -> list[str]:
        response = chat(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"""
                    Page content:
                    \"\"\"
                    {page_content}
                    \"\"\"
                    """,
                },
            ],
            options={
                "temperature": 0.0,
                "num_ctx": 8192,
                "num_predict": 256,
                "top_p": 0.9,
                "repeat_penalty": 1.1,
            },
        )
        raw = response.message.content.strip()
        return raw

    def read_pdf_by_page(self, path_pdf: str):
        doc = fitz.open(path_pdf)
        results_list = []
        result_text = ""

        for page_index, page in enumerate(
            tqdm(doc, desc="Reading PDF pages", unit="page"), start=1
        ):
            text = page.get_text().strip()
            ingredient = {
                "page": page_index,
                "text": self.extract_features_clean(self.extract_page_topics(text)),
            }
            results_list.append(ingredient)
            result_text += " " + str(ingredient)

        return results_list, result_text

class PDFPageStructure:
    def __init__(self, model_name):
        self.model_name = model_name

    def run_gemini(
        self,
        user_prompt: str,
    ) -> dict:
        """
        Send text to Gemini and return parsed JSON output.
        """

        model = genai.GenerativeModel(
            model_name=self.model_name,
        )

        response = model.generate_content(
            user_prompt,
            generation_config={
                "temperature": 0,
                "response_mime_type": "application/json",
            },
        )

        raw = response.text.strip()
        return raw

    def isolated_pages(self, list_indicators, context, descriptions):
        user_prompt = f"""
        You are given a list of indicators and a document structure.
        Each page id describes the main information covered on that page.

        Your task is to identify, for EACH indicator, the pages that are most likely to contain information relevant to that indicator.

        It is normal and expected that the same page may be relevant to multiple indicators.
        If an indicator includes a description with indicator explanation, you MUST use that description as the primary semantic reference when identifying relevant pages, and prioritize pages that explicitly match the described concept over pages that only loosely relate by title.

        Indicators:
        {list_indicators}

        Document structure:
        {context}

        Description:
        {descriptions}

        Output format (STRICT):
        Return ONLY a single JSON object in the following structure:

        {{
        "<indicator_1>": {{
            "thinking": "<Why these pages are relevant to this indicator>",
            "page_list": ["x", "y"]
        }},
        "<indicator_2>": {{
            "thinking": "<Why these pages are relevant to this indicator>",
            "page_list": ["a"]
        }}
        }}

        Do NOT output anything outside the JSON.
        """
        return self.run_gemini(user_prompt=user_prompt)


pipeline_options = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True,
)
doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
        )
    }
)

class PageFocus:
    def __init__(self, model_name: str):
        self.model_name = model_name

    def run_gemini(self, user_prompt):
        model = genai.GenerativeModel(
            model_name=self.model_name,
        )
        response = model.generate_content(
            user_prompt,
            generation_config={"temperature": 0},
        )
        return response.text.strip()

    def parse_llm_json(self, raw_text: str):
        """
        Extract JSON object from Gemini output and parse into Python dict.
        """
        # Find first JSON object inside {...}
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)

        if not match:
            raise ValueError("No JSON object found in LLM output")

        json_str = match.group(0)

        return json.loads(json_str)

    def docling_parse_text(self, num_page: int, path_pdf: str):
        # Lấy tổng số page của PDF
        doc = fitz.open(path_pdf)
        doc.close()
        result = doc_converter.convert(source=path_pdf, page_range=(num_page, num_page))

        return result.document.export_to_markdown()

    def expand_page_list(self, page_list, total_pages):
        """
        Expand page list with ±1 window, deduplicate, sort.

        Input:  ["3", "4", "12"]
        Output: ["2","3","4","5","11","12","13"]
        """

        pages = set()

        for p in page_list:
            p = int(p)

            for x in (p - 1, p, p + 1):
                if x < 1:
                    continue
                if total_pages is not None and x > total_pages:
                    continue
                pages.add(x)

        return [str(p) for p in sorted(pages)]

    def decide(self, page_content: str, indicator: str, indicator_description: str):
        """
        Extract indicator-related information from a given page.

        Output JSON:
        {
          "<page_id>": "<extracted relevant content>" OR null
        }
        """

        user_prompt = f"""
        You are an expert financial analyst.

        Task:
        From the page content below, extract ONLY the information that is directly relevant
        to the requested financial indicator.

        Indicator:
        {indicator}

        Description:
        {indicator_description}

        Rules:
        - If the page contains relevant indicator information, extract the exact sentence(s),
        number(s), ratio(s), or explanation related to it.
        - If the page contains NO relevant information, return null for this page.
        - Output must be valid JSON only.
        - Do NOT add explanations or extra text.

        Output Format:
        {{
        "<page_id>": "<extracted content>" OR null
        }}

        Page Content:
        {page_content}
        """

        return self.parse_llm_json(self.run_gemini(user_prompt))
    
    def run(self, indicator, descriptions, page_list, total_pages, path_pdf):
        if indicator not in descriptions:
            description_ = ""
        else:
            description_ = descriptions[indicator]
            
        check = set()
        focus_data = {}
        focus_page = []
        focus_content = []
        for page in page_list:
            expanded_pages = self.expand_page_list([page], total_pages)
            content = ""
            for page_num in expanded_pages:
                if page_num in check:
                    continue
                check.add(page_num)
                content_ = self.docling_parse_text(int(page_num), path_pdf)
                content = content + F"---PAGE: {page_num}---" + "\n" + content_ + "\n\n"
            res = self.decide(page_content=content, indicator=indicator, indicator_description=description_)
            for key, value in res.items():
                if value is not None:
                    focus_data[key] = value
                    focus_page.append(key)
                    focus_content.append(value)
        content_text = "\n".join(focus_content)

        return focus_data, focus_page, focus_content, content_text
    
class GenAnswerByPageContext:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.focus = PageFocus(model_name=model_name)

    def run_gemini(self, user_prompt):
        model = genai.GenerativeModel(
            model_name=self.model_name,
        )
        response = model.generate_content(
            user_prompt,
            generation_config={
                "temperature": 0,
            },
        )
        raw = response.text.strip()
        return raw

    def gen_answers(self, isolation_dic, descriptions):
        outputs = {}

        for indicator, meta in isolation_dic.items():
            outputs[indicator] = {}
            outputs[indicator]["information"] = {}

            context = ""

            for path, page_list in meta.items():
                if path not in outputs[indicator]["information"]:
                    outputs[indicator]["information"][path] = {
                        "page_list": [],
                        "context": "",
                        "context_vs_index": None
                    }
                    
                if len(page_list) != 0:
                    total_pages = fitz.open(path).page_count
                    res_focus = self.focus.run(indicator, descriptions, page_list, total_pages, path)
                    outputs[indicator]["information"][path]["context_vs_index"] = res_focus[0]
                    outputs[indicator]["information"][path]["page_list"] = res_focus[1]
                    context = res_focus[3]
                    outputs[indicator]["information"][path]["context"] = context

            user_prompt = f"""
            You are an expert financial analyst. 
            Based on the context provided, please extract information related to producted indicator.
            If the indicator includes a description or explanation, you MUST treat it as a strict definition and use it as the primary semantic reference when extracting information from the context.
            The context may not contain information relevant to the indicator.
            If the required information is NOT explicitly stated in the context,
            you MUST respond exactly with: "NO INFORMATION".

            EACH INDICATOR HAS ITS OWN DESCRIPTION.
            ANSWER MUST BE IN ENGLISH.

            NOT: AUTO GENERATE ANYTHING THAT IS NOT IN THE CONTEXT.
  
            
            Indicator: {indicator}

            Descriptions:
            {descriptions.get(indicator, "")}

            Context:
            {context}
            """
            answer = self.run_gemini(user_prompt)
            outputs[indicator]["answer"] = answer

        return outputs

class PDFRretrieval:
    def __init__(self, mini_model, big_model):
        self.mini_model = mini_model
        self.big_model = big_model
        self.pdf_processor = PDFProcessor(model_name=mini_model)
        self.pdf_page_structure = PDFPageStructure(model_name=big_model)
        self.gen_answer_by_page_context = GenAnswerByPageContext(model_name=big_model)

    def run(self, file_paths, list_indicators, descriptions, type_report, target_site):
        use_paths = file_paths[target_site][type_report]
        TREES = {}
        ISOLATED_DIC = {}
        full_time = 0
        print("Start processing PDF pages - Gentree")
        time_gentree = 0

        for path in use_paths:
            start = time.time()
            data = self.pdf_processor.read_pdf_by_page(path)
            TREES[path] = data[1]
            end = time.time()
            time_value = end - start
            time_gentree += time_value

            print(f"Gentree: {path}", round(time_value, 2), "seconds")
        print("--" * 20)
        print("Total Gentree time:", round(time_gentree, 2), "seconds")
        print()

        full_time = full_time + time_gentree

        print("Start isolated")
        start = time.time()

        for indicator in list_indicators:
            ISOLATED_DIC[indicator] = {}

        for path, context in TREES.items():
            result = self.pdf_page_structure.isolated_pages(
                list_indicators, context, descriptions
            )
            isolation_json = json.loads(result)
            for key_indicator, value in isolation_json.items():
                if path not in ISOLATED_DIC[key_indicator]:
                    ISOLATED_DIC[key_indicator][path] = []
                ISOLATED_DIC[key_indicator][path].extend(map(int, value["page_list"]))

        end = time.time()
        isolated_time = end - start
        full_time = full_time + isolated_time
        print("Total isolated pages: ", round(isolated_time, 2), "seconds")
        print()

        print("Generate answer from context")
        start = time.time()
        RESULTS = self.gen_answer_by_page_context.gen_answers(
            ISOLATED_DIC, descriptions
        )
        end = time.time()
        gen_answer_time = end - start
        full_time = full_time + gen_answer_time
        print(
            "Total generate answer from context: ", round(gen_answer_time, 2), "seconds"
        )
        print()

        print("------Finished processing------")
        print("Total run time:", round(full_time, 2), "seconds")

        for indicator, value in RESULTS.items():
            for p, v in value["information"].items():
                ISOLATED_DIC[indicator][p] = v["page_list"]
                
        return RESULTS, TREES, ISOLATED_DIC


def make_safe_filename(url: str) -> str:
    """
    Generate unique, readable filename from URL
    Example:
    https://ssl4.eir-parts.net/doc/6920/tdnet/2691142/00.pdf
    -> 6920_tdnet_2691142_00.pdf
    """
    parsed = urlparse(url)
    parts = parsed.path.strip("/").split("/")

    # Remove leading 'doc' if exists
    if parts and parts[0] == "doc":
        parts = parts[1:]

    filename = "_".join(parts)

    # Keep safe characters only
    filename = re.sub(r"[^a-zA-Z0-9._-]", "_", filename)

    if not filename.lower().endswith(".pdf"):
        filename += ".pdf"

    return filename


def download_reports(report_urls, save_dir="reports", timeout=30):
    """
    Download list of report URLs to local machine
    and return list of saved file paths
    """
    os.makedirs(save_dir, exist_ok=True)
    saved_files = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/pdf",
        "Referer": "https://ssl4.eir-parts.net/",
    }

    for url in report_urls:
        url = str(url)

        try:
            filename = make_safe_filename(url)
            save_path = os.path.join(save_dir, filename)

            # Skip if already downloaded
            if os.path.exists(save_path):
                print(f"⏭️ Skipped (exists): {filename}")
                saved_files.append(save_path)
                continue

            print(f"⬇️ Downloading: {filename}")

            r = requests.get(
                url,
                stream=True,
                timeout=timeout,
                headers=headers,
            )
            r.raise_for_status()

            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:                                               
                        f.write(chunk)

            saved_files.append(save_path)
            print(f"✅ Saved: {save_path}")

        except Exception as e:
            print(f"❌ Failed: {url}")
            print(f"   Reason: {e}")

    return saved_files

✅ Google API configured successfully


# Report data prepare

In [11]:
target_site = "hd.eneos.co.jp"
save_dir = "data/" + target_site
report_urls = [
    "https://ssl4.eir-parts.net/doc/5020/ir_material_for_fiscal_ym9/190074/00.pdf",
    "https://www.hd.eneos.co.jp/esgdb/pdf/system_governance_report.pdf"
]
saved_report = download_reports(report_urls, save_dir = save_dir)


file_paths = {
    target_site: {
        "ir_report": [
            "data/hd.eneos.co.jp/00.pdf",
        ],
        "governance_report": ["data/hd.eneos.co.jp/system_governance_report.pdf"],
    }
}

⏭️ Skipped (exists): 5020_ir_material_for_fiscal_ym9_190074_00.pdf
⏭️ Skipped (exists): esgdb_pdf_system_governance_report.pdf


# Extraction

## 📈Ir Report

#### Shareholder Return Policy

In [ ]:
type_report = "ir_report"
list_indicators = [
    "Shareholder Return Policy",
]

descriptions = {}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_ir, TREES_ir, ISOLATED_DIC_ir = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)
print(TREES_ir)
print()
print("====="*100)
print()
print(ISOLATED_DIC_ir)
print()
print("====="*100)
print()
print(RESULTS_ir)

#### IR Event Frequency

In [ ]:
type_report = "ir_report"
list_indicators = [
    "IR Event Frequency",
]

descriptions = {}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_ir, TREES_ir, ISOLATED_DIC_ir = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)
print(TREES_ir)
print()
print("====="*100)
print()
print(ISOLATED_DIC_ir)
print()
print("====="*100)
print()
print(RESULTS_ir)

## 🏛️ Governance Report

#### Major Shareholder Structure

In [ ]:
type_report = "governance_report"
list_indicators = ["Major Shareholder Structure"]

descriptions = {}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_governance, TREES_governance, ISOLATED_DIC_governance = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)

print(TREES_governance)
print()
print("====="*100)
print()
print(ISOLATED_DIC_governance)
print()
print("====="*100)
print()
print(RESULTS_governance)

#### Controlling Shareholder

In [ ]:
type_report = "governance_report"
list_indicators = ["Controlling Shareholder"]

descriptions = {
    "Controlling Shareholder": """Determine if a controlling shareholder exists (do NOT assume the largest shareholder is controlling).
                                1. Controlling shareholder: Yes / No
                                If Yes: 
                                    - 2.: Name and ownership %
                                If No: 
                                    - 2.: Brief justification and name and ownership percentage of the largest major shareholder mentioned
                                """
}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_governance, TREES_governance, ISOLATED_DIC_governance = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)

print(TREES_governance)
print()
print("====="*100)
print()
print(ISOLATED_DIC_governance)
print()
print("====="*100)
print()
print(RESULTS_governance)

#### Parent Company Relationship

In [ ]:
type_report = "governance_report"
list_indicators = ["親会社との関係性"]

descriptions = {
    "親会社との関係性":  """
                    Find the section that explicitly discusses the company's relationship with its parent company (親会社) or holding company (持株会社).

                    This indicator includes:
                    - Whether a parent company exists
                    - Capital relationship (資本関係), ownership ratio (持株比率, 議決権比率)
                    - Controlling shareholder (支配株主)
                    - Transactions with the parent company (親会社との取引, 関連当事者取引)
                    - Independence of management (経営の独立性)
                    - Dispatch of directors/employees from parent (役員派遣)
                    Return only pages where parent-company relationship is explicitly mentioned.
                """
}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_governance, TREES_governance, ISOLATED_DIC_governance = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)

print(TREES_governance)
print()
print("====="*100)
print()
print(ISOLATED_DIC_governance)
print()
print("====="*100)
print()
print(RESULTS_governance)

#### Governance Report Content

In [ ]:
type_report = "governance_report"
list_indicators = ["Governance Report Content"]

descriptions = {
    "Governance Report Content": """
    This includes content on: governance structure, governance policy, board composition, committees, audit system, internal control, shareholder relations, disclosure policy, cross-shareholding, compensation system, risk management, and governance improvement actions.
    """
}

main = PDFRretrieval(mini_model="qwen2.5:3b", big_model="gemini-2.5-flash")
RESULTS_governance, TREES_governance, ISOLATED_DIC_governance = main.run(
    file_paths=file_paths,
    list_indicators=list_indicators,
    descriptions=descriptions,
    type_report=type_report,
    target_site=target_site,
)

print(TREES_governance)
print()
print("====="*100)
print()
print(ISOLATED_DIC_governance)
print()
print("====="*100)
print()
print(RESULTS_governance)

Start processing PDF pages - Gentree


Reading PDF pages: 100%|██████████| 18/18 [01:16<00:00,  4.26s/page]


Gentree: data/hd.eneos.co.jp/system_governance_report.pdf 76.64 seconds
----------------------------------------
Total Gentree time: 76.64 seconds

Start isolated
Total isolated pages:  18.58 seconds

Generate answer from context
Total generate answer from context:  317.91 seconds

------Finished processing------
Total run time: 413.13 seconds


In [ ]:
# RESULTS_governance['Governance Report Content']['information']['data/hd.eneos.co.jp/system_governance_report.pdf']["page_list"]

In [ ]:
# RESULTS_governance['Governance Report Content']['information']['data/hd.eneos.co.jp/system_governance_report.pdf']['context_vs_index']